# Cover Studio (Google Colab)

Runs the Cover Studio web UI on Colab. It turns a pop song into a **piano cover MIDI**, and lets you regenerate it with different settings and compare the takes.

1. In the menu, choose **Runtime → Change runtime type** and select a **GPU (e.g. T4)**
2. Run the cell below and click **"▶ Open Cover Studio"** (if it does not open in a new tab, run "Open inside this cell" below it to show the UI inside the notebook)

Everything else happens in the UI: upload audio → transcription ([tsumugi](https://github.com/anime-song/tsumugi)) → cover generation → comparison → MIDI / WAV download.

- The first run downloads libraries (PyTorch etc.) and models (tsumugi and the cover model), which takes several minutes
- Keep this notebook tab open while you use the UI (the runtime stops if you close it or leave it idle for a while)
- Transcription takes a few minutes per song, but only once. Add a transcribed MIDI together with the audio to skip it
- The cover model weights ([anime-song/tsumugi-piano-cover](https://huggingface.co/anime-song/tsumugi-piano-cover)) are licensed under **CC BY-NC 4.0 (non-commercial)**

In [ ]:
#@title ▶ Start Cover Studio
#@markdown Running this cell shows an "▶ Open Cover Studio" link below.
#@markdown
#@markdown Save projects (audio, source MIDI, takes) to `MyDrive/cover_studio` on Google Drive and continue next time:
USE_DRIVE = True  #@param {type:"boolean"}
REPO_URL = "https://github.com/anime-song/tsumugi-piano-cover"  #@param {type:"string"}
TSUMUGI_URL = "https://github.com/anime-song/tsumugi"  #@param {type:"string"}

import os
import socket
import subprocess
import sys
import time
from pathlib import Path

REPO = Path("/content/tsumugi-piano-cover")
TSUMUGI = Path("/content/tsumugi")
PYTHON = TSUMUGI / ".venv" / "bin" / "python"
PORT = 8100
LOG = Path("/content/cover_studio_server.log")


def running():
    with socket.socket() as s:
        return s.connect_ex(("127.0.0.1", PORT)) == 0


def run(what, cmd, cwd=None):
    """Run a command. Hide its output unless it fails"""
    print(f"▶ {what}", end="", flush=True)
    start = time.time()
    r = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    if r.returncode != 0:
        print(" … failed\n" + (r.stdout + r.stderr)[-4000:])
        raise RuntimeError(f"Failed at: {what} (see the output above)")
    print(f" … done ({time.time() - start:.0f} s)")


if subprocess.run(["nvidia-smi"], capture_output=True).returncode != 0:
    print("⚠ No GPU. Choose a GPU under Runtime → Change runtime type to make transcription and generation much faster")

if USE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    DATA = Path("/content/drive/MyDrive/cover_studio")
else:
    DATA = Path("/content/cover_studio_data")
(DATA / "projects").mkdir(parents=True, exist_ok=True)

if not running():
    for url, path in ((REPO_URL, REPO), (TSUMUGI_URL, TSUMUGI)):
        if path.exists():
            run(f"Updating {path.name}", ["git", "-C", str(path), "pull", "-q", "--ff-only"])
        else:
            run(f"Downloading {path.name}", ["git", "clone", "-q", "--depth", "1", url, str(path)])
    run("Installing uv", [sys.executable, "-m", "pip", "install", "-q", "uv"])
    run("Installing tsumugi's libraries (a few minutes the first time)", ["uv", "sync", "--locked", "--extra", "stem"], cwd=TSUMUGI)
    run(
        "Installing Cover Studio's libraries",
        ["uv", "pip", "install", "-q", "--python", str(PYTHON), "-r", str(REPO / "cover_studio" / "requirements.txt")],
    )
    print("▶ Starting the server (the UI and the models are downloaded automatically the first time)")
    subprocess.Popen(
        [
            str(PYTHON), "-m", "cover_studio", "--no-browser", "--port", str(PORT),
            "--root", str(DATA / "projects"), "--tsumugi-dir", str(TSUMUGI),
            # Use the cover model published on Hugging Face (downloaded on the first generation)
            "--pretrained-dir", "", "--checkpoints-dir", "",
        ],
        cwd=REPO, stdout=open(LOG, "a"), stderr=subprocess.STDOUT, start_new_session=True,
        env={**os.environ, "PYTHONUNBUFFERED": "1"},
    )
    for _ in range(180):
        if running():
            break
        time.sleep(1)
    else:
        raise RuntimeError(f"The server did not start. Check the log cell below or {LOG}")

from google.colab import output

print("✅ Ready. Open the link below (if it does not open, run \"Open inside this cell\" below)")
output.serve_kernel_port_as_window(PORT, anchor_text="▶ Open Cover Studio")

In [ ]:
#@title (If the link does not open) Open inside this cell
#@markdown If "▶ Open Cover Studio" does not open a new tab, shows a blank page, or shows an error, run this cell to show the UI inside the notebook.
#@markdown Run "Start Cover Studio" above first.
#@markdown
#@markdown Height of the UI (pixels):
HEIGHT = 900  #@param {type:"integer"}

import socket

from google.colab import output

PORT = 8100

with socket.socket() as s:
    if s.connect_ex(("127.0.0.1", PORT)) != 0:
        raise RuntimeError('The server is not running. Run "Start Cover Studio" above first')

print("If MIDI downloads do not work, the covers are also saved as MyDrive/cover_studio/projects/<song>/takes/<take>/cover.mid on Google Drive (when USE_DRIVE is on)")
output.serve_kernel_port_as_iframe(PORT, height=HEIGHT)

In [ ]:
#@title (Troubleshooting) Show the server log
!tail -n 100 /content/cover_studio_server.log

In [ ]:
#@title (Troubleshooting) Stop the server
#@markdown After stopping, run "Start Cover Studio" again to restart it.
import subprocess
import time

subprocess.run(["pkill", "-f", "cover_studio --no-browser"])
time.sleep(2)
print("Stopped the server. Run the cell above again to restart it")